# 🎞️ ROTBOTS — Assemble
## Combine Videos + Audio + Credits into Final Video

---

The final step! FFmpeg combines everything:

1. **Normalize** all clips to same format
2. **Generate rolling credits** from source attributions
3. **Concatenate** scenes + credits
4. **Overlay** narration audio
5. **Export** the final video

> **🤔 Think about:** How does the order of scenes change the meaning? Who gets credited?

---

## 🔧 Setup

In [ ]:
# SETUP
import json
import subprocess
import textwrap
from pathlib import Path
from IPython.display import display, Markdown, Video, HTML

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
VIDEOS_DIR = WORK_DIR / 'videos'
AUDIO_DIR = WORK_DIR / 'audio'
TEMP_DIR = Path('/content/temp_assembly')
TEMP_DIR.mkdir(exist_ok=True)

video_files = sorted(VIDEOS_DIR.glob('scene_*.mp4')) if VIDEOS_DIR.exists() else []
audio_files = sorted(AUDIO_DIR.glob('narration_*.mp3')) if AUDIO_DIR.exists() else []

print(f'🎬 Videos: {len(video_files)} clips')
print(f'🎙️ Audio: {len(audio_files)} narrations')

# Load essay for credits data
essay = None
essay_file = WORK_DIR / 'essay_script.json'
if essay_file.exists():
    with open(essay_file) as f: essay = json.load(f)
    print(f'📖 Essay: {essay.get("title", "Untitled")}')

if not video_files: print('\n❌ No videos found! Run 05_Generate.ipynb first.')
else: print('\n✅ Ready to assemble!')

In [ ]:
# FFmpeg helpers
def get_duration(path):
    try:
        r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)], capture_output=True, text=True, timeout=10)
        return float(r.stdout.strip())
    except: return 5.0

def run_ffmpeg(cmd, desc=''):
    if desc: print(f'   {desc}...', end=' ', flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if r.returncode == 0:
        if desc: print('✅')
        return True
    else:
        print(f'❌ {r.stderr[:200]}')
        return False

print('✅ Helpers loaded')

---
## 🎞️ Station 6: Assemble

In [ ]:
# ============================================================
# STEP 1: Normalize all videos to same format
# ============================================================
print('🔧 Step 1: Normalizing video formats...')

normalized_videos = []
for v in video_files:
    out = TEMP_DIR / v.name
    cmd = [
        'ffmpeg', '-y', '-i', str(v),
        '-vf', 'scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black',
        '-r', '24', '-pix_fmt', 'yuv420p',
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
        '-an', str(out)
    ]
    if run_ffmpeg(cmd, f'Normalizing {v.name}'):
        normalized_videos.append(out)

print(f'\n✅ {len(normalized_videos)} videos normalized')

In [ ]:
# ============================================================
# STEP 2: Generate Rolling Credits
# ============================================================
print('🎬 Step 2: Generating credits...')

# Build credits text
title = essay.get('title', 'Untitled') if essay else 'Untitled'
sources = essay.get('credits', {}).get('sources', []) if essay else []

credits_lines = [
    title,
    '',
    'Generated by ROTBOTS',
    'Content Machine Workshop',
    '',
    '— Sources —',
]
for s in sources:
    # Truncate long URLs for display
    display_source = s[:60] + '...' if len(s) > 60 else s
    credits_lines.append(display_source)

credits_lines.extend([
    '',
    '— Pipeline —',
    'LLM: Groq (Llama 3.3)',
    'Video: fal.ai',
    'Voice: Edge-TTS (Microsoft)',
    'Composition: FFmpeg',
    '',
    'LI-MA Transformation Digital Art 2026',
    'Amsterdam',
])

# Credits settings
CREDITS_DURATION = 8  # seconds
CREDITS_WIDTH = 1280
CREDITS_HEIGHT = 720
FONT_SIZE = 28
LINE_HEIGHT = 42

# Calculate scroll: text starts at bottom, scrolls to top
total_text_height = len(credits_lines) * LINE_HEIGHT
scroll_distance = CREDITS_HEIGHT + total_text_height
scroll_speed = scroll_distance / CREDITS_DURATION  # pixels per second

# Build FFmpeg drawtext filter chain
# Each line is a separate drawtext filter with y position offset
drawtext_filters = []
for i, line in enumerate(credits_lines):
    if not line:  # empty line = spacer
        continue
    # Escape special characters for FFmpeg
    safe_line = line.replace("'", "\u2019").replace(':', '\\:').replace('%', '%%')
    y_offset = i * LINE_HEIGHT
    # y = starts at bottom (h) + line offset, scrolls up by scroll_speed * t
    drawtext_filters.append(
        f"drawtext=text='{safe_line}':fontsize={FONT_SIZE}:fontcolor=white:"
        f"x=(w-text_w)/2:y=h+{y_offset}-{scroll_speed:.1f}*t"
    )

filter_chain = ','.join(drawtext_filters)

credits_path = TEMP_DIR / 'credits.mp4'
cmd = [
    'ffmpeg', '-y',
    '-f', 'lavfi', '-i', f'color=c=black:s={CREDITS_WIDTH}x{CREDITS_HEIGHT}:d={CREDITS_DURATION}:r=24',
    '-vf', filter_chain,
    '-pix_fmt', 'yuv420p',
    '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
    str(credits_path)
]

if run_ffmpeg(cmd, 'Generating rolling credits'):
    print(f'   Duration: {CREDITS_DURATION}s')
    print(f'   Lines: {len(credits_lines)}')
    print(f'   Sources credited: {len(sources)}')
else:
    print('   ⚠️ Credits generation failed, continuing without credits')
    credits_path = None

In [ ]:
# ============================================================
# STEP 3: Concatenate all videos + credits
# ============================================================
print('🎬 Step 3: Concatenating scenes + credits...')

concat_file = TEMP_DIR / 'concat.txt'
with open(concat_file, 'w') as f:
    for v in normalized_videos:
        f.write(f"file '{v}'\n")
    # Append credits at the end
    if credits_path and credits_path.exists():
        f.write(f"file '{credits_path}'\n")
        print(f'   Including {len(normalized_videos)} scenes + credits')
    else:
        print(f'   Including {len(normalized_videos)} scenes (no credits)')

concat_output = TEMP_DIR / 'concatenated.mp4'
cmd = [
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', str(concat_file),
    '-c', 'copy',
    str(concat_output)
]

if run_ffmpeg(cmd, 'Concatenating'):
    duration = get_duration(concat_output)
    print(f'   Total duration: {duration:.1f}s')

In [ ]:
# ============================================================
# STEP 4: Mix narration audio (if available)
# ============================================================
final_output = WORK_DIR / 'final_video.mp4'

if audio_files:
    print('🎙️ Step 4: Adding narration audio...')

    # Concatenate all narration audio
    audio_concat_file = TEMP_DIR / 'audio_concat.txt'
    with open(audio_concat_file, 'w') as f:
        for a in audio_files:
            f.write(f"file '{a}'\n")

    combined_audio = TEMP_DIR / 'combined_narration.mp3'
    run_ffmpeg([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
        '-i', str(audio_concat_file),
        '-c', 'copy', str(combined_audio)
    ], 'Combining narration')

    # Overlay narration on video
    cmd = [
        'ffmpeg', '-y',
        '-i', str(concat_output),
        '-i', str(combined_audio),
        '-c:v', 'copy',
        '-c:a', 'aac', '-b:a', '192k',
        '-map', '0:v:0', '-map', '1:a:0',
        '-shortest',
        str(final_output)
    ]
    run_ffmpeg(cmd, 'Mixing video + narration')
else:
    print('ℹ️ No narration — video only')
    import shutil
    shutil.copy2(concat_output, final_output)

if final_output.exists():
    size_mb = final_output.stat().st_size / (1024 * 1024)
    duration = get_duration(final_output)
    print(f'\n✅ Final video created!')
    print(f'   📁 {final_output}')
    print(f'   📐 Duration: {duration:.1f}s')
    print(f'   💾 Size: {size_mb:.1f}MB')

---
## 🎬 Watch Your Video!

In [ ]:
# PLAY FINAL VIDEO
if final_output.exists():
    display(Markdown('# 🎬 Your AI-Generated Video'))
    display(Markdown(f'**{title}**'))
    display(Markdown(f'*Created by the ROTBOTS content machine*\n\n---'))
    try:
        display(Video(str(final_output), embed=True, width=720))
    except:
        print(f'Cannot preview inline.')
        print(f'Download from Google Drive: My Drive/rotbots_workshop/final_video.mp4')
else:
    print('❌ No final video found')

In [ ]:
# DOWNLOAD (optional)
DOWNLOAD_LOCALLY = False  # Set True to download

if DOWNLOAD_LOCALLY and final_output.exists():
    from google.colab import files
    files.download(str(final_output))
    print('📥 Download started!')
else:
    print(f'📁 Video on Google Drive: My Drive/rotbots_workshop/final_video.mp4')
    print(f'   Set DOWNLOAD_LOCALLY = True above to download.')

---
## 🎤 Station 7: Screening & Critique

Share your video with the group!

**Discussion:**
- How does your video compare to others?
- Where is the bias in this pipeline?
- Who is the "author"?
- Could you tell it was AI-generated?
- What does this mean for digital art preservation?

---
## 📊 Pipeline Summary

In [ ]:
# PIPELINE SUMMARY
print('📊 ROTBOTS Pipeline Summary')
print('=' * 60)

steps = []
if (WORK_DIR / 'summaries.json').exists():
    with open(WORK_DIR / 'summaries.json') as f: s = json.load(f)
    steps.append(f'📥 Sources: {len(s.get("sources", []))} scraped & summarized')

if essay:
    ch_count = len(essay.get('chapters', []))
    sec_count = sum(len(c.get('sections',[])) for c in essay.get('chapters',[]))
    total_words = sum(len(s.get('narration','').split()) for c in essay.get('chapters',[]) for s in c.get('sections',[]))
    steps.append(f'✍️ Essay: "{essay.get("title", "Untitled")}" — {ch_count} chapters, {sec_count} sections, {total_words} words')

if (WORK_DIR / 'storyboard.json').exists():
    with open(WORK_DIR / 'storyboard.json') as f: sb = json.load(f)
    steps.append(f'🎬 Storyboard: {len(sb)} scenes')

if (WORK_DIR / 'prompts.json').exists():
    with open(WORK_DIR / 'prompts.json') as f: p = json.load(f)
    steps.append(f'🎥 Video prompts: {len(p)} T2V prompts')

steps.append(f'🎙️ Narration: {len(audio_files)} audio files')
steps.append(f'🎬 Video clips: {len(video_files)} generated/uploaded')
steps.append(f'📜 Credits: {len(sources)} sources attributed')

if final_output.exists():
    dur = get_duration(final_output)
    size = final_output.stat().st_size / (1024*1024)
    steps.append(f'🎞️ Final video: {dur:.1f}s, {size:.1f}MB')

for step in steps: print(f'   {step}')

print(f'\n{"=" * 60}')
print(f'\n🤖 All generated by AI from a single topic.')
print(f'   Human input: choosing the topic and pressing Play.')

---
## 🏁 That's it!

You've built and operated an AI content machine.

From a single topic, the pipeline:
1. Scraped and summarized sources
2. Wrote a structured essay with narration
3. Translated text into visual video prompts
4. Generated a synthetic voice
5. Created AI video clips
6. Generated rolling credits with source attribution
7. Assembled everything into a finished video

**Every step involved automated decisions — about what's important, what looks good, what sounds right, and how to tell the story.**

The question isn't whether AI can make content. It clearly can.

The question is: **what does that mean?**

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026, Amsterdam*